# Self-learning homophonic carver — Colab T4
Clones the repo, installs deps, and runs SFT warm-start + best-of-N self-improvement against the prosody-aware reward. Runtime → Change runtime type → **T4 GPU**.

In [ ]:
!nvidia-smi -L  # confirm a GPU is attached

In [ ]:
# --- clone the repo ---
REPO = 'grundlagen/Lingua-Sound-Wave'
BRANCH = 'claude/phrase-weave-multiword'
GH_TOKEN = ''  # <- paste a GitHub token IF the repo is private; leave '' if public
url = f'https://{GH_TOKEN+"@" if GH_TOKEN else ""}github.com/{REPO}.git'
!rm -rf repo && git clone --depth 1 -b $BRANCH $url repo
%cd repo/research/homophone-bench

In [ ]:
# --- system + python deps (espeak-ng is required by the matcher/prosody) ---
!apt-get -qq install -y espeak-ng >/dev/null
!pip -q install transformers trl datasets accelerate panphon wordfreq numpy

In [ ]:
# --- sanity: the reward runs (prosody sound x French-validity) ---
!python selflearn/reward.py

In [ ]:
# --- self-learning run (T4-sized) ---
%cd selflearn
!python train_selflearn.py --base Qwen/Qwen2.5-1.5B-Instruct --rounds 4 --k 8 --keep_thresh 0.55

In [ ]:
# --- try the self-learned carver ---
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
tok = AutoTokenizer.from_pretrained('./homophonic-carver')
m = AutoModelForCausalLM.from_pretrained('./homophonic-carver', torch_dtype=torch.float16, device_map='auto')
for en in ['the little cat', 'my friend loves the sea', 'gentle rain']:
    p = f'Rewrite the English so it sounds the same when read aloud in French: {en}'
    ids = tok.apply_chat_template([{'role':'user','content':p}], add_generation_prompt=True, return_tensors='pt').to(m.device)
    out = m.generate(ids, max_new_tokens=24, do_sample=False)
    print(en, '->', tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip())

Save the model to Drive to keep it, then iterate: raise `--keep_thresh` each run so it chases ever-better carves. The reward is the linguistics (prosody stress/contour + real-French validity); the model is its own teacher.